# Daste, Pierré et al. (2025), Go/NoGo experiment, from dandiset 000785

Paper: Daste, Pierré et al., "Projection-specific Routing of Odor Information in the Olfactory
Cortex", bioRxiv preprint doi:10.64898/2025.12.12.694045 (version 1). The claims file
(`claims/000785/10.64898-2025.12.12.694045.yaml`) was written from the preprint's full text
flattened from its JATS XML: main text, Methods and supplementary legends, without the contents
of Tables 2 to 8.

Dataset: DANDI:000785, version **0.260520.1603** (doi:10.48324/dandi.000785/0.260520.1603). The
paper cites no version. The two published versions, 0.260519.1753 and 0.260520.1603, hold the
same 61 files with the same blobs (checked through the DANDI API before the analysis), so the
most recent is pinned. Files are streamed, never downloaded.

This notebook covers the Go/NoGo sessions (experiments GNG_2021_dataset and GNG_2023b_dataset):
behavior, pseudo-population decoding and the tracked neurons. The counts and passive experiments
are in `reproduce_daste_2025_passive.ipynb`.

| Claim | Tested here |
|---|---|
| fig4c-behavior-learning, fig4c-anticipatory-licking | tested |
| fig4def-identity-decoding-gng, fig4g-valence-any, fig4h-valence-timing, fig4ij-valence-correct-trials, figs8-generalized-valence-correct | tested (day 3) |
| discussion-differences-grow | tested (days 1 to 3) |
| tracked-neuron-counts | tested |
| figs10d-slope-no-difference, figs10e-sparseness-slope-no-difference, fig5c-cv-higher-obp | tested (tracked neurons) |
| fig5e-transferability, fig5f-stability | tested (tracked neurons, per mouse) |

Mice. The paper's Go/NoGo experiment has 7 mice (4 OB-p, 3 mPFC-p); the dandiset has 8, with a
fourth mPFC mouse, sub-668. The main analyses use the 7 mice without sub-668, an assumption: it is
the only Go/NoGo mouse without tracking identifiers, and it has 24 to 58 cells per session
against 98 to 361 for the others. Its performance does not explain its absence (the Methods
exclude "Mice that failed to reach >= 60% correct after 3 days"; its day 3 performance is printed
below). The behavior, licking and day 3 decoding claims are also shown with all 8 mice.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import pingouin as pg
import warnings
import pcx

warnings.filterwarnings("ignore", message="The least populated class")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
S = pcx.survey_all()
S["subject"] = S.subject.astype(str)
SES = []
for exp in ["GNG_2021_dataset", "GNG_2023b_dataset"]:
    ses = pcx.load_experiment(S, exp)
    dates = sorted({s["meta"]["start"][:10] for s in ses})
    for s in ses:
        s["day"] = dates.index(s["meta"]["start"][:10])     # the code's day_num: rank of the date - 1
        s["subject"] = s["meta"]["subject"]
        s["proj"] = s["meta"]["projection_area"]
        SES.append(s)
SES = [s for s in SES if s["day"] <= 3]                    # the code ignores day 4 of GNG_2023b
for s in SES:
    tr = s["trials"]
    s["good"] = (~tr.is_bad_trial.astype(bool)).to_numpy()
    if s["day"] > 0:
        tr["outcome_relabel"] = pcx.hit_relabel(tr)
        tr["valence"] = np.where(tr.stim_annotation.str.startswith("go"), "reward", "none")
        s["keep"] = s["good"] & tr.stim_annotation.notna().to_numpy()
print("sessions:", len(SES))

## Data check

In [ ]:
rows = []
for s in SES:
    tr = s["trials"]
    r = dict(subject=s["subject"], proj=s["proj"], day=s["day"], task=s["meta"]["task"], cells=len(s["cells"]),
             red=int(s["cells"].red.sum()), with_muse=int((s["cells"].muse >= 0).sum()), trials=len(tr),
             bad=int(tr.is_bad_trial.sum()), fs=round(s["fs"], 3), X_min=round(float(s["X"].min()), 1),
             X_max=round(float(s["X"].max()), 1))
    if s["day"] > 0:
        oc = tr.outcome.value_counts()
        r.update(hit=oc.get("hit", 0), miss=oc.get("miss", 0), CR=oc.get("correct_rejection", 0), FA=oc.get("false_alarm", 0),
                 relabeled=int((tr.outcome != tr.outcome_relabel).sum()),
                 reward_after_onset=f"{(tr.reward_onset - tr.stim_onset).min():.2f}-{(tr.reward_onset - tr.stim_onset).max():.2f}",
                 per_CS=tr.stim_annotation.value_counts().min())
    rows.append(r)
SAN = pd.DataFrame(rows).sort_values(["proj", "subject", "day"])
print(SAN.to_string(index=False))
tr = [s for s in SES if s["day"] == 1][0]["trials"]
print("onset within trial (s):", sorted({round(float(v), 2) for s in SES for v in (s["trials"].stim_onset_timestamp - s["trials"].start_time)}))
print("odor duration (s):", sorted({round(float(v), 2) for s in SES for v in (s["trials"].stim_offset_timestamp - s["trials"].stim_onset_timestamp)}))
L = np.concatenate([np.concatenate(s["trials"].lick_onset.to_list()) for s in SES if s["day"] > 0])
print(f"lick onsets (s from trial start): {L.min():.2f} to {L.max():.2f}, n = {len(L)}")
print("hit trials without reward:", sum(int(((s['trials'].outcome == 'hit') & s['trials'].reward_onset.isna()).sum()) for s in SES if s['day'] > 0),
      "| rewarded trials not hits:", sum(int(((s['trials'].outcome != 'hit') & s['trials'].reward_onset.notna()).sum()) for s in SES if s['day'] > 0))
print("trial type matches CS annotation:", all(((s["trials"].trial_type == "GO") == s["trials"].stim_annotation.str.startswith("go")).all() for s in SES if s["day"] > 0))
n_clip = {f"sub-{s['subject']} day {s['day']}": s["n_clipped"] for s in SES if s["n_clipped"]}
print("trials whose -5 to 15 s window runs past the recording's ends (edge frames repeated):", n_clip if n_clip else 0,
      "of", sum(len(s["trials"]) for s in SES))
print("largest distance between an onset and the plane-0 frame taken for it (s):", round(max(s["onset_error_s"] for s in SES), 3))
fpt = {s["n_frames_session"] // len(s["trials"]) for s in SES}
ok_frame = all(((s["on_idx"] - s["trials"].stim_onset_frame.to_numpy().astype(int))
                == np.arange(len(s["trials"])) * (s["n_frames_session"] // len(s["trials"]))).all() for s in SES)
print(f"frames per trial: {fpt}; the onset frame taken from the timestamps equals the file's stim_onset_frame "
      f"(frame within the trial) in every trial of every session: {ok_frame}")
ex = [s for s in SES if s["subject"] == "2456" and s["day"] == 3][0]["trials"]
cols_t = ["start_time", "stim_onset", "stim_onset_timestamp", "stim_onset_frame", "reward_onset", "reward_onset_timestamp", "outcome"]
print("one trial (sub-2456, day 3, trial 1):"); print(ex.loc[[1], cols_t].to_string())
print("   its first licks, lick_onset:", np.round(ex.lick_onset.iloc[1][:3], 3), "| lick_onset_timestamp:", np.round(ex.lick_onset_timestamp.iloc[1][:3], 3))
d_lick = max(float(np.max(np.abs(np.asarray(lt) - st - np.asarray(lo)))) for s in SES if s["day"] > 0
             for lo, lt, st in zip(s["trials"].lick_onset, s["trials"].lick_onset_timestamp, s["trials"].start_time) if len(lo))
d_on = max(float((s["trials"].stim_onset_timestamp - s["trials"].start_time - s["trials"].stim_onset).abs().max()) for s in SES if s["day"] > 0)
print(f"lick_onset and stim_onset are times from the trial start: largest |lick_onset_timestamp - start_time - lick_onset| = {d_lick:.4f} s, "
      f"|stim_onset_timestamp - start_time - stim_onset| = {d_on:.4f} s")

## Recordings deposited more than once

Every file of the dandiset is compared with every other, plane by plane: the correlation of the
mean images, and whether the first 300 frames of summed raw fluorescence are identical. Planes
within a file are compared too (the paper's code skips planes without red cells because they
"tend to have issues, such as duplicated planes"). Sessions with identical trial stimulus
sequences are listed.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(12) as ex:
    FP = dict(zip(S.path, ex.map(pcx.fingerprint, S.asset_id)))
keys = [(path, i) for path in FP for i in FP[path]["planes"]]
imgs = np.stack([FP[k[0]]["planes"][k[1]]["img"] for k in keys])
imgs = (imgs - imgs.mean(1, keepdims=True)) / imgs.std(1, keepdims=True)
Cimg = imgs @ imgs.T / imgs.shape[1]
same_trace = []
for a in range(len(keys)):
    for b in range(a + 1, len(keys)):
        ta, tb = FP[keys[a][0]]["planes"][keys[a][1]]["trace"], FP[keys[b][0]]["planes"][keys[b][1]]["trace"]
        if len(ta) == len(tb) and np.array_equal(ta, tb):
            same_trace.append((keys[a], keys[b]))
iu = np.triu_indices(len(keys), 1)
pairs = pd.DataFrame(dict(a=[keys[i] for i in iu[0]], b=[keys[j] for j in iu[1]], r=Cimg[iu]))
pairs["same_file"] = [x[0] == y[0] for x, y in zip(pairs.a, pairs.b)]
print("planes compared:", len(keys))
print("plane pairs with identical summed raw traces (first 300 frames):", same_trace)
print("plane pairs within one file with mean-image r > 0.95:")
print(pairs[pairs.same_file & (pairs.r > 0.95)].to_string(index=False))
print("highest within-file mean-image r:", round(float(pairs[pairs.same_file].r.max()), 3),
      "| highest between-file r:", round(float(pairs[~pairs.same_file].r.max()), 3))
print("plane pairs between files with mean-image r > 0.99:", int((~pairs.same_file & (pairs.r > 0.99)).sum()))
seq = pd.Series({p: FP[p]["stims"] for p in FP})
dups = seq[seq.duplicated(keep=False)]
print("files sharing an identical trial stimulus sequence:")
for sq, grp in dups.groupby(dups.map(hash)):
    print("  ", list(grp.index))

## fig4c-behavior-learning

> Go/NoGo performance improves across training days, with no difference between OB-p and mPFC-p
> labelled mice, reaching 73.7% correct trials on day 3.

Per mouse and task day, percent correct = (hits + correct rejections) / scored CS trials,
excluding bad trials, with hits (a) as scored in the NWB `outcome` column and (b) relabeled
(reward within 10 s after the delay). One-way repeated-measures ANOVA on days and mixed ANOVA
(days within, projection between), with pingouin.

In [ ]:
beh = []
for s in SES:
    if s["day"] == 0:
        continue
    tr = s["trials"][s["keep"]]
    for rd, col in [("a: as scored", "outcome"), ("b: relabeled", "outcome_relabel")]:
        beh.append(dict(subject=s["subject"], proj=s["proj"], day=s["day"], reading=rd,
                        pct_correct=100 * tr[col].isin(["hit", "correct_rejection"]).mean(), n=len(tr)))
BEH = pd.DataFrame(beh)
print(BEH.pivot_table(index=["proj", "subject"], columns=["reading", "day"], values="pct_correct").round(1).to_string())
for mice in ["7 mice (without sub-668)", "all 8 mice"]:
    d0 = BEH if mice == "all 8 mice" else BEH[BEH.subject != "668"]
    for rd in ["a: as scored", "b: relabeled"]:
        d = d0[d0.reading == rd]
        rm = pg.rm_anova(d, dv="pct_correct", within="day", subject="subject")
        mx = pg.mixed_anova(d, dv="pct_correct", within="day", between="proj", subject="subject")
        print(f"\n{mice}, {rd}: day 3 mean {d[d.day == 3].pct_correct.mean():.2f}%; "
              f"RM ANOVA F = {rm.F.iloc[0]:.3f}, p = {rm['p_unc'].iloc[0]:.4f}")
        print(mx[["Source", "F", "p_unc"]].round(4).to_string(index=False))
        print("   day means:", d.groupby("day").pct_correct.mean().round(2).to_dict())

**Result: not reproduced.** With 7 mice the day 3 mean is 76.9% (hits as scored) and 76.4% (relabeled), within 73.7 +/- 7.37, and the day means rise (69.6, 72.5, 76.9%). The day effect is not significant: repeated-measures F = 1.571 (p = 0.248) and 2.452 (p = 0.128), against the paper's F = 4.375 (p = 0.0374), and mixed-ANOVA day p = 0.246 and 0.118. Projection (p = 0.66) and interaction (p >= 0.26) are not significant, as in the paper. With all 8 mice and relabeled hits F = 3.742 (p = 0.0499), still outside 4.375 +/- 0.4375, with mixed-ANOVA day p = 0.029.

## fig4c-anticipatory-licking

> Anticipatory licking in response to the rewarded odorants increases across training.

Lick rate (licks per second, from `lick_onset`) on CS+ trials in (a) the delay, 2 to 5 s after
odor onset, and (b) odor and delay, 0 to 5 s.

In [ ]:
lk = []
for s in SES:
    if s["day"] == 0:
        continue
    tr = s["trials"][s["keep"]]
    tr = tr[tr.trial_type == "GO"]
    for rd, (a, b) in [("a: 2-5 s", (2, 5)), ("b: 0-5 s", (0, 5))]:
        n = [np.sum((np.asarray(l) - on >= a) & (np.asarray(l) - on < b)) for l, on in zip(tr.lick_onset, tr.stim_onset)]
        lk.append(dict(subject=s["subject"], proj=s["proj"], day=s["day"], reading=rd, rate=np.mean(n) / (b - a)))
LK = pd.DataFrame(lk)
print(LK.pivot_table(index=["proj", "subject"], columns=["reading", "day"], values="rate").round(2).to_string())
for mice in ["7 mice (without sub-668)", "all 8 mice"]:
    d0 = LK if mice == "all 8 mice" else LK[LK.subject != "668"]
    for rd in ["a: 2-5 s", "b: 0-5 s"]:
        m = d0[d0.reading == rd].groupby("day").rate.mean()
        print(f"{mice}, {rd}: mean lick rate by day {m.round(3).to_dict()}; day 3 > day 1: {m[3] > m[1]}")

**Result: reproduced.** With 7 mice, the mean CS+ lick rate 2 to 5 s after onset is 0.49, 1.07 and 1.90 licks/s on days 1 to 3, and 0.50, 0.99 and 1.68 over 0 to 5 s. Each of the 8 mice licks more on day 3 than on day 1 under both readings.

## sub-668

The mouse the dandiset adds to the paper's 7. Its cells, red cells, tracking identifiers and
day 3 performance against the Methods' exclusion threshold of 60% correct.

In [ ]:
print(SAN[SAN.subject == "668"][["day", "cells", "red", "with_muse"]].to_string(index=False))
print(BEH[(BEH.subject == "668") & (BEH.day == 3)][["reading", "pct_correct"]].to_string(index=False))
print("lowest day 3 performance among the other 7:", BEH[(BEH.subject != "668") & (BEH.day == 3)].groupby("reading").pct_correct.min().round(1).to_dict())

## Pseudo-populations and the clock check

Per task day, cells pooled across mice: OB-p (red cells of OB mice), mPFC-p (red cells of mPFC
mice) and any-p (all cells of all mice). Trials are matched by CS (`stim_annotation`, go1, go2,
nogo1, nogo2) and by order of that CS, truncated to the smallest count across mice. For correct
trials, the correct trials (relabeled hits and correct rejections) are matched in the same way.

The clock check decodes CS identity from the any-p pseudo-population in a baseline window (2 s
before onset) and in the odor phase: if the traces share the trial clock, the baseline is at
chance and the odor phase is not.

In [ ]:
def gng_pseudo(day, group, correct=False, exclude=("668",)):
    ss = [s for s in SES if s["day"] == day and s["subject"] not in exclude]
    masks = []
    for s in ss:
        if group == "any":
            masks.append(np.ones(len(s["cells"]), bool))
        else:
            masks.append(((s["cells"].projection == group)).to_numpy())
    keeps = []
    for s in ss:
        k = s["keep"].copy()
        if correct:
            k &= s["trials"].outcome_relabel.isin(["hit", "correct_rejection"]).to_numpy()
        keeps.append(k)
    tab, P = pcx.pseudo_population(ss, masks, "stim_annotation", keeps)
    return tab.stim_annotation.to_numpy(), P, ss[0]["t"]

GROUPS = ["OB", "mPFC", "any"]
PH = list(pcx.PHASES)
for d in [1, 2, 3]:
    for g in GROUPS:
        for corr in [False, True]:
            ann, P, t = gng_pseudo(d, g, corr)
            print(f"day {d} {g:4s} correct={corr!s:5s}: {P.shape[0]} cells, {len(ann)} trials, per CS {pd.Series(ann).value_counts().min()}")
rng_c = np.random.default_rng(4000)
ann, P, t = gng_pseudo(3, "any")
chk = pcx.decode_gng(P, ann, t, 85, 50, rng_c, schemes=("identity",), phases=[(-2, 0), (0, 2)], aggs=("mean",))
print("clock check, day 3 any-p identity, 50 repeats: baseline (-2 to 0 s) "
      f"{100 * chk['identity', 'mean'][:, 0].mean():.1f}%, odor phase {100 * chk['identity', 'mean'][:, 1].mean():.1f}% (chance 25%)")
for s in SES:
    if s["day"] != 3 or s["subject"] == "668":
        continue
    a = s["trials"][s["keep"]].stim_annotation.to_numpy()
    X = s["X"][:, s["keep"]]
    r = pcx.decode_gng(X, a, s["t"], 10**9, 5, rng_c, schemes=("identity",), phases=[(-2, 0), (0, 2)], aggs=("mean",))["identity", "mean"].mean(0)
    print(f"   per session, sub-{s['subject']} day 3, all its cells: baseline {100 * r[0]:.1f}%, odor {100 * r[1]:.1f}%")

### The clock check per session

In the day 3 check above two mPFC mice (sub-2456 and sub-2457) decode no better in the odor phase
than at baseline with all their cells. To tell a misaligned clock from a late or weak response,
each session of days 1 to 3 is decoded (CS identity, all its cells, 5 repeats) in 1 s windows
from 5 s before to 15 s after onset, and the onset-aligned mean of the normalized traces (all
cells, all kept trials) of each day 3 session is printed frame by frame from 2 s before to 4 s
after onset. A misaligned clock would show decoding above chance before onset, or a response
time course shifted as a whole; the frame of the onset is also checked against the files'
`stim_onset_frame` in the data check.

In [ ]:
rng_cs = np.random.default_rng(4010)
W1 = [(a, a + 1) for a in range(-5, 15)]
rows = []
for s in SES:
    if s["day"] == 0 or s["subject"] == "668":
        continue
    a = s["trials"][s["keep"]].stim_annotation.to_numpy()
    r = pcx.decode_gng(s["X"][:, s["keep"]], a, s["t"], 10**9, 5, rng_cs, schemes=("identity",), phases=W1, aggs=("mean",))["identity", "mean"].mean(0)
    rows.append(dict(proj=s["proj"], subject=s["subject"], day=s["day"], **{f"{w[0]}": round(100 * v) for w, v in zip(W1, r)}))
CLK = pd.DataFrame(rows).sort_values(["proj", "subject", "day"])
print("identity accuracy (%) per session, 1 s windows labelled by their start (s from onset); chance 25%")
print(CLK.to_string(index=False))
pre = CLK[[f"{a}" for a in range(-5, 0)]].to_numpy()
print(f"before onset: {pre.min()} to {pre.max()}% (mean {pre.mean():.1f}); 0-1 s: {CLK['0'].min()} to {CLK['0'].max()}%; "
      f"1-2 s: {CLK['1'].min()} to {CLK['1'].max()}%; 2-3 s: {CLK['2'].min()} to {CLK['2'].max()}%")
print("\nonset-aligned mean normalized trace, all cells and kept trials, day 3 (columns: frames from the onset frame, 0.22 s each)")
t_c = SES[0]["t"]
rows = []
for s in SES:
    if s["day"] != 3 or s["subject"] == "668":
        continue
    m = s["X"][:, s["keep"]].mean((0, 1))
    i0 = s["n_pre"]
    base = m[(s["t"] >= -4) & (s["t"] < 0)]
    after = np.where((s["t"] >= 0) & (m > base.mean() + 3 * base.std()))[0]
    rows.append(dict(proj=s["proj"], subject=s["subject"], first_s_above_baseline_3sd=round(float(s["t"][after[0]]), 2) if len(after) else np.nan,
                     **{f"{k}": round(float(m[i0 + k]), 2) for k in range(-9, 19)}))
print(pd.DataFrame(rows).sort_values(["proj", "subject"]).to_string(index=False))

The per-session check is consistent with one clock for traces and trials in every session,
including sub-2456 and sub-2457, though it does not prove it. In every session the onset frame
taken from the timestamps is the file's own `stim_onset_frame` (data check), decoding before
onset stays within the range of shuffled labels (20 to 39%; shuffled means 26 to 29% and 95th
percentiles 36 to 39%, below), and
decoding rises only after onset. In sub-2456 and sub-2457 it rises later and less than in the
other mice on all three days: 34 to 51% in 1 to 2 s after onset, against 40 to 85% for the
other mice (56% or more in all sessions but sub-2400 day 1). Their mean traces still change from the onset frame on (sub-2457
rises from 0.80 to 0.89 by 0.9 s, as the OB-p mice do). A whole-trial offset would shift the
pre-onset windows too, and these do not decode. So these two mice respond later or more weakly in
the odor phase, which the data do not explain further; their share of the day 3 mPFC-p cells
(61 of 93) is shown by mouse under fig4def-identity-decoding-gng.

In [ ]:
def run_day(day, n_rep, seed, correct=False, schemes=("identity", "valence", "generalized"), exclude=("668",), groups=GROUPS):
    rng = np.random.default_rng(seed)
    out = {}
    for g in groups:
        ann, P, t = gng_pseudo(day, g, correct, exclude)
        out[g] = pcx.decode_gng(P, ann, t, 85, n_rep, rng, schemes=schemes)
    return out

DEC3 = run_day(3, 100, 4301)
DEC3C = run_day(3, 100, 4302, correct=True)
DEC3_8 = run_day(3, 100, 4303, exclude=())
DEC3C_8 = run_day(3, 100, 4304, correct=True, exclude=())

def table(dec, scheme, agg, chance):
    rows = []
    for g in dec:
        A = dec[g][scheme, agg]
        for pi, ph in enumerate(PH):
            tt = stats.ttest_1samp(A[:, pi], chance, alternative="greater")
            rows.append(dict(group=g, phase=ph, mean=100 * A[:, pi].mean(), sd=100 * A[:, pi].std(),
                             p_vs_chance_bonf=min(1, 4 * tt.pvalue)))
    T = pd.DataFrame(rows)
    btw = []
    for pi, ph in enumerate(PH):
        p = pcx.mwu(dec["OB"][scheme, agg][:, pi], dec["mPFC"][scheme, agg][:, pi])
        btw.append(dict(phase=ph, OB_minus_mPFC=100 * (dec["OB"][scheme, agg][:, pi].mean() - dec["mPFC"][scheme, agg][:, pi].mean()),
                        p_OB_vs_mPFC_bonf=min(1, 4 * p)))
    return T, pd.DataFrame(btw)

### Trial counts and empirical chance

The paper tests identity against 25% and valence against 50%. The pseudo-population tables are
not balanced, so the chance level of this procedure is computed: per table, the counts of each
CS and of CS+ trials, and the accuracy of the same decoders (85 cells, mean aggregate) with the
CS labels shuffled across pseudo-trials, 50 repeats. For correct trials a balanced variant is
also run, each CS truncated to the smallest CS count (the paper excludes days 1 and 2 "due to
imbalanced counts of correct trials across animals to properly construct pseudo-populations",
Figure 4I-J legend), decoding valence and generalized valence (identity cannot be
cross-validated with 3 trials per CS).

In [ ]:
rng_n = np.random.default_rng(4050)
null_rows = []
for label, excl in [("7 mice", ("668",)), ("all 8 mice", ())]:
    for corr in [False, True]:
        for g in GROUPS:
            ann, P, t = gng_pseudo(3, g, corr, excl)
            vc = pd.Series(ann).value_counts()
            acc = {sc: [] for sc in ["identity", "valence", "generalized"]}
            for r in range(50):
                d = pcx.decode_gng(P, ann[rng_n.permutation(len(ann))], t, 85, 1, rng_n, aggs=("mean",))
                for sc in acc:
                    acc[sc].append(d[sc, "mean"][0])
            for sc in acc:
                A = np.array(acc[sc])
                null_rows.append(dict(mice=label, correct=corr, group=g, scheme=sc, trials=len(ann), per_CS=vc.to_dict(),
                                      CS_plus_fraction=round(float(np.mean([a.startswith("go") for a in ann])), 3),
                                      **{ph: round(100 * A[:, pi].mean(), 1) for pi, ph in enumerate(PH)},
                                      null_95th=round(100 * np.percentile(A, 95), 1)))
NULL = pd.DataFrame(null_rows)
print("shuffled-label accuracy (%), day 3, mean over 50 repeats per phase, and the 95th percentile over all phases")
print(NULL.drop(columns=["per_CS", "trials", "CS_plus_fraction"]).to_string(index=False))
print(NULL.drop_duplicates(["mice", "correct"])[["mice", "correct", "trials", "per_CS", "CS_plus_fraction"]].to_string(index=False))

def balanced(ann):
    """Indices keeping the first n pseudo-trials of each CS, n the smallest CS count."""
    n = pd.Series(ann).value_counts().min()
    return np.concatenate([np.where(ann == a)[0][:n] for a in sorted(set(ann))])

def decode_balanced(P, ann, n_rep, rng, shuffle=False):
    """Valence and generalized valence on a balanced table: dict (scheme, agg) -> repeats x phases."""
    from sklearn.model_selection import StratifiedKFold
    out = {(sc, ag): np.zeros((n_rep, len(PH))) for sc in ["valence", "generalized"] for ag in ["mean", "max"]}
    for r in range(n_rep):
        a = ann[rng.permutation(len(ann))] if shuffle else ann
        v = np.array([x.startswith("go") for x in a])
        cells = rng.choice(P.shape[0], 85, replace=False)
        folds = {"valence": list(StratifiedKFold(5, shuffle=True, random_state=int(rng.integers(2**31))).split(np.zeros(len(v)), v))}
        for ag in ["mean", "max"]:
            for pi, R in enumerate(pcx.phase_features(P[cells], t_c, list(pcx.PHASES.values()), ag)):
                for sc in ["valence", "generalized"]:
                    out[sc, ag][r, pi] = pcx.decode_scheme(R, a, sc, folds)
    return out

rng_b = np.random.default_rng(4060)
BAL = {}
for g in ["OB", "mPFC"]:
    ann, P, t = gng_pseudo(3, g, True)
    k = balanced(ann)
    BAL[g] = decode_balanced(P[:, k], ann[k], 100, rng_b)
    BAL[g, "null"] = decode_balanced(P[:, k], ann[k], 50, rng_b, shuffle=True)
print(f"\nbalanced correct trials, 7 mice: {len(k)} pseudo-trials, {pd.Series(ann[k]).value_counts().to_dict()}")
for sc in ["valence", "generalized"]:
    for ag in ["mean", "max"]:
        for g in ["OB", "mPFC"]:
            A, N = BAL[g][sc, ag], BAL[g, "null"][sc, ag]
            print(f"   {sc:11s} {ag:4s} {g:4s} accuracy {np.round(100 * A.mean(0), 1)} | shuffled {np.round(100 * N.mean(0), 1)} | "
                  f"p against 50% (Bonferroni 4): {[f'{min(1, 4 * stats.ttest_1samp(A[:, pi], 0.5, alternative='greater').pvalue):.2g}' for pi in range(4)]}")
        print(f"   {sc:11s} {ag:4s} OB against mPFC, two-sided Mann-Whitney, Bonferroni 4: "
              f"{[f'{min(1, 4 * pcx.mwu(BAL['OB'][sc, ag][:, pi], BAL['mPFC'][sc, ag][:, pi])):.2g}' for pi in range(4)]}")

## fig4def-identity-decoding-gng

> In expert mice (day 3), OB-p pseudo-populations decode odor identity above chance in all four
> task phases, peaking early (delay) and declining later, whereas mPFC-p pseudo-populations are
> near chance early and improve later in the trial.

Day 3, 4 CS (chance 25%), 85 cells, 100 repeats, aggregate (a) mean and (b) maximum over the
phase. One-sample one-sided t-tests against 25%, Bonferroni over 4 phases.

In [ ]:
for label, dec in [("7 mice", DEC3), ("all 8 mice", DEC3_8)]:
    for agg in ["mean", "max"]:
        T, Bt = table(dec, "identity", agg, 0.25)
        ob = T[T.group == "OB"].set_index("phase")
        mp = T[T.group == "mPFC"].set_index("phase")
        cond = dict(OB_all_above=bool((ob.p_vs_chance_bonf < 0.05).all()), OB_peak=ob["mean"].idxmax(),
                    OB_late_below_delay=bool(ob.loc["act-late", "mean"] < ob.loc["delay", "mean"]),
                    mPFC_two_lowest=sorted(mp["mean"].nsmallest(2).index.tolist()), mPFC_peak=mp["mean"].idxmax())
        print(f"\n{label}, aggregate {agg}")
        print(T.pivot(index="group", columns="phase", values="mean")[PH].round(2).to_string())
        print(T.pivot(index="group", columns="phase", values="p_vs_chance_bonf")[PH].to_string())
        print(Bt.to_string(index=False))
        print(cond)
print("\nshuffled-label identity accuracy, 7 mice (%):")
print(NULL[(NULL.mice == "7 mice") & ~NULL.correct & (NULL.scheme == "identity")][["group"] + PH + ["null_95th"]].to_string(index=False))
# by mouse: each mouse's own red cells on day 3, all of them, 20 repeats
rng_m = np.random.default_rng(4070)
rows = []
for s in SES:
    m = s["cells"].red.to_numpy()
    if s["day"] != 3 or m.sum() < 2:
        continue
    a = s["trials"][s["keep"]].stim_annotation.to_numpy()
    r = pcx.decode_gng(s["X"][m][:, s["keep"]], a, s["t"], 10**9, 20, rng_m, schemes=("identity",), aggs=("mean",))["identity", "mean"].mean(0)
    rows.append(dict(proj=s["proj"], subject=s["subject"], red_cells=int(m.sum()), **{ph: round(100 * v, 1) for ph, v in zip(PH, r)}))
print("\nby mouse, day 3, identity from the mouse's own red cells (mean aggregate, 20 repeats, chance 25%):")
print(pd.DataFrame(rows).sort_values(["proj", "subject"]).to_string(index=False))
print("day 3 mPFC-p cells per mouse (7 mice):", {s["subject"]: int(s["cells"].red.sum()) for s in SES if s["day"] == 3 and s["proj"] == "mPFC" and s["subject"] != "668"})

**Result: reproduced.** With 7 mice and the mean aggregate, OB-p reaches 54.5, 75.0, 71.0 and 58.6% in odor, delay, act-early and act-late, above 25% in every phase (corrected p <= 5e-71), highest in the delay and lower in act-late. mPFC-p reaches 36.4, 52.7, 65.8 and 60.2%: lowest in odor and delay, highest in act-early. The maximum aggregate and all 8 mice give the same pattern. mPFC-p is also above chance in every phase (odor 36.4%, corrected p = 5e-43), so "near chance" holds only in the sense of lowest. With labels shuffled, this procedure gives 26 to 29% (95th percentile 36 to 39%), a little above the paper's 25%; every mean above is higher than that. OB-p exceeds mPFC-p in odor, delay and act-early (corrected p <= 3.1e-12) and not in act-late (p = 0.12, mean aggregate). By mouse, from each mouse's own red cells, each of the three mPFC mice decodes least in the odor phase (sub-2456 31.4%, sub-2457 30.0%, sub-644 35.5%), so the pattern is not carried by sub-2456 and sub-2457 alone; they supply 61 of the 93 day 3 mPFC-p cells.

## fig4g-valence-any

> Generalized odor valence can be decoded above chance from a general PCx (any-p)
> pseudo-population in all task phases in expert mice.

In [ ]:
for label, dec in [("7 mice", DEC3), ("all 8 mice", DEC3_8)]:
    for agg in ["mean", "max"]:
        T, _ = table(dec, "generalized", agg, 0.5)
        a = T[T.group == "any"]
        print(f"{label}, {agg}: any-p generalized valence by phase", dict(zip(a.phase, a["mean"].round(2))),
              "| Bonferroni p:", dict(zip(a.phase, a.p_vs_chance_bonf.map(lambda v: f"{v:.3g}"))))

**Result: reproduced.** Any-p generalized valence reaches 54.1, 69.3, 64.5 and 60.2% in the four phases (7 mice, mean aggregate), all above 50% (corrected p <= 9.1e-07). The maximum aggregate and all 8 mice also give corrected p < 0.05 in every phase.

## fig4h-valence-timing

> In expert mice OB-p pseudo-populations encode generalized valence in the odor and delay
> phases but fall to chance in the action phases, while mPFC-p pseudo-populations encode valence
> in the later but not the earliest phase; the two differ in all four phases.

In [ ]:
for label, dec in [("7 mice", DEC3), ("all 8 mice", DEC3_8)]:
    for agg in ["mean", "max"]:
        T, Bt = table(dec, "generalized", agg, 0.5)
        sig = T.set_index(["group", "phase"]).p_vs_chance_bonf < 0.05
        cond = dict(OB_odor=sig["OB", "odor"], OB_delay=sig["OB", "delay"], OB_actlate_not=not sig["OB", "act-late"],
                    mPFC_odor_not=not sig["mPFC", "odor"], mPFC_delay=sig["mPFC", "delay"], mPFC_ae=sig["mPFC", "act-early"],
                    mPFC_al=sig["mPFC", "act-late"], n_phases_between=int((Bt.p_OB_vs_mPFC_bonf < 0.05).sum()))
        print(f"\n{label}, aggregate {agg}")
        print(T.pivot(index="group", columns="phase", values="mean")[PH].round(2).to_string())
        print(T.pivot(index="group", columns="phase", values="p_vs_chance_bonf")[PH].to_string())
        print(Bt.to_string(index=False))
        print(cond)

**Result: not reproduced.** With 7 mice and the mean aggregate, OB-p reaches 62.6, 66.9, 47.7 and 56.2%: above 50% in odor and delay, at chance in act-early (corrected p = 1.0), but above chance in act-late (corrected p = 1.7e-25), where the claim has it at chance. mPFC-p reaches 52.0, 60.1, 59.8 and 64.2%, above chance in delay and both action phases and also in odor (corrected p = 4.1e-10). Under the maximum aggregate mPFC-p is at chance in odor (49.4%, p = 1.0), but OB-p is still above chance in act-late (56.7%, p = 4.7e-26). OB-p and mPFC-p differ in all four phases under both aggregates. The early OB-p and late mPFC-p shape holds; the pattern of above-chance phases does not.

## fig4ij-valence-correct-trials

> Restricted to correct trials, OB-p pseudo-populations encode valence well early and much less
> later, mPFC-p pseudo-populations encode it more accurately late than early, and the two differ
> in all four phases.

Valence (unconstrained), correct trials only, 5-fold stratified cross-validation on valence. Shown on
the table matched as for all trials and on the table balanced over the 4 CS (see "Trial counts
and empirical chance").

In [ ]:
for label, dec in [("7 mice", DEC3C), ("all 8 mice", DEC3C_8)]:
    for agg in ["mean", "max"]:
        T, Bt = table(dec, "valence", agg, 0.5)
        ob = T[T.group == "OB"].set_index("phase")["mean"]
        mp = T[T.group == "mPFC"].set_index("phase")
        cond = dict(OB_early_gt_late=bool(min(ob["odor"], ob["delay"]) > max(ob["act-early"], ob["act-late"])),
                    mPFC_late_gt_early=bool(min(mp.loc["act-early", "mean"], mp.loc["act-late", "mean"]) > max(mp.loc["odor", "mean"], mp.loc["delay", "mean"])),
                    mPFC_odor_not_above=bool(mp.loc["odor", "p_vs_chance_bonf"] >= 0.05),
                    n_phases_between=int((Bt.p_OB_vs_mPFC_bonf < 0.05).sum()))
        print(f"\n{label}, aggregate {agg}")
        print(T.pivot(index="group", columns="phase", values="mean")[PH].round(2).to_string())
        print(T.pivot(index="group", columns="phase", values="p_vs_chance_bonf")[PH].to_string())
        print(Bt.to_string(index=False))
        print(cond)
print("\nshuffled-label valence accuracy, correct trials (%):")
print(NULL[NULL.correct & (NULL.scheme == "valence")][["mice", "group"] + PH + ["null_95th", "CS_plus_fraction"]].to_string(index=False))
print("\nbalanced reading (each CS truncated to 3 correct pseudo-trials, 12 in all; chance from the shuffle above):")
for ag in ["mean", "max"]:
    ob, mp = 100 * BAL["OB"]["valence", ag].mean(0), 100 * BAL["mPFC"]["valence", ag].mean(0)
    p_mp = [min(1, 4 * stats.ttest_1samp(BAL["mPFC"]["valence", ag][:, pi], 0.5, alternative="greater").pvalue) for pi in range(4)]
    p_bt = [min(1, 4 * pcx.mwu(BAL["OB"]["valence", ag][:, pi], BAL["mPFC"]["valence", ag][:, pi])) for pi in range(4)]
    cond = dict(OB_early_gt_late=bool(min(ob[0], ob[1]) > max(ob[2], ob[3])), mPFC_late_gt_early=bool(min(mp[2], mp[3]) > max(mp[0], mp[1])),
                mPFC_odor_not_above=bool(p_mp[0] >= 0.05), n_phases_between=int(sum(p < 0.05 for p in p_bt)))
    print(f"   {ag}: OB {np.round(ob, 1)}, mPFC {np.round(mp, 1)}, mPFC p against 50% {[f'{p:.2g}' for p in p_mp]}, "
          f"OB against mPFC p {[f'{p:.2g}' for p in p_bt]}")
    print("  ", cond)

**Result: reproduced, resting on an assumption** (correct-trial pseudo-populations balanced over the 4 CS). As first run, with trials matched by CS and order and truncated per CS, the correct-trial table has 30 trials, 77% of them CS+ (17, 6, 4 and 3 per CS), and decoding with shuffled labels gives 65 to 69% for valence instead of the paper's 50%: that procedure is not the paper's. Under it OB-p reaches 83.5, 83.7, 79.4 and 81.9% (7 mice, mean), mPFC-p 72.6, 80.7, 80.4 and 82.7%, and the two differ in 2 of 4 phases. Balancing each CS to the smallest count, 3 correct trials, leaves 12 trials and a shuffled level of 42 to 51%. Under that reading, with the maximum aggregate, OB-p reaches 78.9, 70.4, 55.8 and 55.2%, higher in odor and delay than in both action phases; mPFC-p reaches 31.4, 47.3, 66.5 and 86.8%, higher in both action phases, and is not above 50% in odor (corrected p = 1); OB-p and mPFC-p differ in all 4 phases (corrected p <= 1.6e-12). With the mean aggregate all holds except act-early (p = 1). These accuracies rest on 12 pseudo-trials.

## figs8-generalized-valence-correct

> Generalized valence decoding in correct trials differs between OB-p and mPFC-p
> pseudo-populations in the odor and act-early phases but not in the delay and act-late phases.

In [ ]:
for label, dec in [("7 mice", DEC3C), ("all 8 mice", DEC3C_8)]:
    for agg in ["mean", "max"]:
        T, Bt = table(dec, "generalized", agg, 0.5)
        sig = dict(zip(Bt.phase, Bt.p_OB_vs_mPFC_bonf < 0.05))
        ok = sig["odor"] and sig["act-early"] and not sig["delay"] and not sig["act-late"]
        print(f"{label}, {agg}:")
        print(Bt.to_string(index=False))
        print("   pattern (odor and act-early differ, delay and act-late do not):", ok)
print("\nshuffled-label generalized valence accuracy, correct trials (%):")
print(NULL[NULL.correct & (NULL.scheme == "generalized")][["mice", "group"] + PH + ["null_95th"]].to_string(index=False))
print("\nbalanced reading (each CS truncated to 3 correct pseudo-trials, 12 in all):")
for ag in ["mean", "max"]:
    p_bt = [min(1, 4 * pcx.mwu(BAL["OB"]["generalized", ag][:, pi], BAL["mPFC"]["generalized", ag][:, pi])) for pi in range(4)]
    dif = 100 * (BAL["OB"]["generalized", ag].mean(0) - BAL["mPFC"]["generalized", ag].mean(0))
    sig = dict(zip(PH, [p < 0.05 for p in p_bt]))
    print(f"   {ag}: OB minus mPFC {np.round(dif, 1)}, p {[f'{p:.2g}' for p in p_bt]}; pattern holds: "
          f"{sig['odor'] and sig['act-early'] and not sig['delay'] and not sig['act-late']}")

**Result: not reproduced.** OB-p and mPFC-p differ in the odor phase (corrected p = 4.3e-33), as the claim says, but also in the delay (p = 2.5e-33, mPFC-p higher by 9.1 points) and in act-late (p = 0.018). Act-early differs under the mean aggregate (p = 1.5e-06) and not under the maximum (p = 0.18). The predicted pattern holds under no reading, with 7 or 8 mice. This table's shuffled level is 61 to 64%, not 50%; with each CS balanced to 3 correct trials (shuffled level 46 to 51%) all four phases differ (corrected p <= 6e-05 under the mean, <= 1.9e-07 under the maximum), so the pattern fails under that reading too.

## discussion-differences-grow

> The differences in timing of identity and valence encoding between OB-p and mPFC-p ensembles
> increase over the course of learning.

Figure S7A and S9 legends: per projection, task phase and day (1 to 3), the accuracies of 50
size-matched pseudo-populations (85 cells); one accuracy is drawn per day and the slope over
days fitted, 1000 times; the null draws the same way with the days shuffled. Two-sided
Mann-Whitney U of observed against null slopes, Bonferroni over the 4 phases. Identity and
generalized valence, aggregates mean and maximum.

In [ ]:
DAYDEC = {d: run_day(d, 50, 4500 + d, schemes=("identity", "generalized"), groups=["OB", "mPFC"]) for d in [1, 2, 3]}
rng_s = np.random.default_rng(4600)
grow = []
days = np.array([1, 2, 3])
for scheme in ["identity", "generalized"]:
    for agg in ["mean", "max"]:
        for g in ["OB", "mPFC"]:
            for pi, ph in enumerate(PH):
                A = np.stack([DAYDEC[d][g][scheme, agg][:, pi] for d in [1, 2, 3]], 1)   # 50 x 3
                draw = A[rng_s.integers(0, 50, (1000, 3)), np.arange(3)]
                obs = np.polyfit(days, draw.T, 1)[0]
                shuf = np.array([rng_s.permutation(row) for row in draw])
                null = np.polyfit(days, shuf.T, 1)[0]
                p = pcx.mwu(obs, null)
                grow.append(dict(scheme=scheme, agg=agg, group=g, phase=ph, day1=100 * A[:, 0].mean(), day2=100 * A[:, 1].mean(),
                                 day3=100 * A[:, 2].mean(), median_slope=100 * np.median(obs), median_null=100 * np.median(null),
                                 p_bonf=min(1, 4 * p)))
GROW = pd.DataFrame(grow)
GROW["positive"] = (GROW.p_bonf < 0.05) & (GROW.median_slope > GROW.median_null)
print(GROW.round(4).to_string(index=False))
for scheme in ["identity", "generalized"]:
    for agg in ["mean", "max"]:
        d = {(r.group, r.phase): bool(r.positive) for r in GROW[(GROW.scheme == scheme) & (GROW["agg"] == agg)].itertuples()}
        ok = (d["OB", "odor"] or d["OB", "delay"]) and (d["mPFC", "act-early"] or d["mPFC", "act-late"])
        print(f"{scheme}, {agg}: OB positive in odor or delay and mPFC positive in an action phase: {ok}")
print("\nOB-p minus mPFC-p mean accuracy (points) by day, the difference the statement is about (not tested by the criterion):")
DIFF = GROW.pivot_table(index=["scheme", "agg", "phase"], columns="group", values=["day1", "day2", "day3"])
for dd in ["day1", "day2", "day3"]:
    DIFF[("diff", dd)] = DIFF[(dd, "OB")] - DIFF[(dd, "mPFC")]
DIFF = DIFF["diff"].reindex(PH, level="phase")
DIFF["abs_grows_1_to_3"] = DIFF.day3.abs() > DIFF.day1.abs()
print(DIFF.round(1).to_string())

**Result: reproduced, resting on an assumption** (sub-668 left out; only the 7-mouse analysis was run here). For identity with the mean aggregate, OB-p accuracy rises over days in the odor (median slope 8.0 points per day) and delay (8.5) phases, and mPFC-p accuracy in act-early (13.7) and act-late (9.4), all above the shuffled null with corrected p < 0.05. The same holds for identity with the maximum and for generalized valence with the mean, but not for generalized valence with the maximum. Both groups also rise in most other phases (mPFC-p identity in odor 2.0 and delay 5.2 points per day), so the criterion's test shows accuracy growing in the named phases, not the OB-p and mPFC-p difference itself growing. The difference itself, which the criterion does not test, grows from day 1 to day 3 for identity in odor (OB-p minus mPFC-p 5.0, 14.3 and 18.0 points, mean aggregate) and delay (15.0, 10.9, 22.2) and shrinks in act-early (29.5, -3.2, 5.7) and act-late (23.1, -1.4, -1.6). With 1000 resampled slopes against 1000 null slopes the Mann-Whitney test is significant for almost any shift, so these p-values say little about size.

## tracked-neuron-counts

> Longitudinal tracking across the three Go/NoGo sessions follows 46 OB-p and 41 mPFC-p neurons
> in 4 OB-p and 3 mPFC-p tagged mice.

The code's criteria (`__determine_omni_criteria__`): a cell is tracked when its identifier
(`subject_global_roi_muse`) is present on days 0, 1, 2 and 3, and labeled when red in at least 3
of its sessions (all of the mouse's sessions, day 4 of GNG_2023b included, as the code counts
them). Counted per projection, with the code's other criteria.

In [ ]:
ALL = []
for exp in ["GNG_2021_dataset", "GNG_2023b_dataset"]:
    sub = S[S.experiment_name == exp].copy()
    sub["day"] = sub.date.rank(method="dense").astype(int) - 1
    for _, r in sub.iterrows():
        f = pcx.open_h5(r.asset_id)
        ps = f["processing/ophys/ImageSegmentation/PlaneSegmentation"]
        if "subject_global_roi_muse" not in ps:
            continue
        ic = ps["iscell"][:, 0] > 0
        ALL.append(pd.DataFrame(dict(subject=r.subject, proj=r.projection_area, day=r.day,
                                     muse=ps["subject_global_roi_muse"][:][ic], persist=ps["num_sessions_persisted_muse"][:][ic],
                                     red=ps["is_redcell_finalized"][:][ic].astype(int))))
ALL = pd.concat(ALL, ignore_index=True)
print("cells with an identifier:", len(ALL), "| duplicated identifiers within a session:",
      int(ALL.duplicated(["subject", "day", "muse"]).sum()))
chk = ALL.groupby(["subject", "muse"]).agg(n=("day", "size"), persist=("persist", "first"))
print("num_sessions_persisted_muse equals the recomputed count:", bool((chk.n == chk.persist).all()))
CC = ALL.sort_values("day").groupby(["subject", "proj", "muse"]).agg(days=("day", lambda x: "|".join(map(str, x))),
                                                                  reds=("red", list)).reset_index()
CC["omni"] = CC.days.str.startswith("0|1|2|3")
CC["nred"] = CC.reds.apply(sum)
CC["max_s"] = CC.subject.map(ALL.groupby("subject").day.nunique())
crit = {"min_3_sessions": CC.nred >= 3, "min_1_session": CC.nred >= 1,
        "first_4_sessions": CC.reds.apply(lambda x: sum(x[:4]) == 4), "all_sessions": CC.nred == CC.max_s}
for name, c in crit.items():
    sel = CC[c & CC.omni]
    print(f"{name:17s}", sel.groupby("proj").size().to_dict(), "| per mouse:", sel.groupby(["proj", "subject"]).size().to_dict())
print("paper: 46 OB-p and 41 mPFC-p, 4 OB-p and 3 mPFC-p mice")
# The code counts red sessions over every session, which includes day 4 of GNG_2023b; the claims file's wording is
# "red in at least 3 of them" (the four sessions, day 0 to 3). Variant: red counted over days 0 to 3 only.
sel4 = CC[CC.omni & (CC.reds.apply(lambda x: sum(x[:4])) >= 3)]
print("variant, red in at least 3 of days 0 to 3:", sel4.groupby("proj").size().to_dict(), "| per mouse:", sel4.groupby(["proj", "subject"]).size().to_dict())
TRK = CC[crit["min_3_sessions"] & CC.omni][["subject", "proj", "muse"]].reset_index(drop=True)

**Result: reproduced.** The code's criterion gives exactly 46 OB-p and 41 mPFC-p tracked neurons. They come from 3 OB-p mice (sub-664 contributes none) and 3 mPFC-p mice. The code counts red sessions over every session, including day 4 of GNG_2023b; counted over days 0 to 3 only, as the claims file words it, the counts are 43 and 37, still within tolerance. The other criteria give 73 and 64 (red in at least 1 session), 35 and 30 (red in the first 4) and 26 and 20 (red in all sessions). The stored persistence counts agree with the identifiers.

## Responses of the tracked neurons

For each tracked neuron and task day, its trials (good CS trials) and normalized fluorescence,
found by its identifier in that day's session.

In [ ]:
TR = {}   # (subject, day) -> dict(X cells x trials x frames in TRK order, trials)
for s in SES:
    if s["day"] == 0 or s["subject"] not in set(TRK.subject):
        continue
    ids = TRK[TRK.subject == s["subject"]].muse.to_numpy()
    pos = pd.Series(np.arange(len(s["cells"])), index=s["cells"].muse.to_numpy())
    TR[s["subject"], s["day"]] = dict(X=s["X"][pos.loc[ids].to_numpy()][:, s["keep"]], trials=s["trials"][s["keep"]].reset_index(drop=True),
                                     t=s["t"], fs=s["fs"], X_all=s["X"][pos.loc[ids].to_numpy()], good=s["good"])
print({k: v["X"].shape for k, v in TR.items()})
t_g = SES[0]["t"]
PHW = list(pcx.PHASES.values())

def phase_means(X, trials, by):
    """Mean over the phase and over the trials of each level of `by`: dict level -> cells x phases."""
    out = {}
    for lev in sorted(trials[by].unique()):
        m = (trials[by] == lev).to_numpy()
        out[lev] = np.stack([X[:, m][:, :, (t_g >= a) & (t_g < b)].mean((1, 2)) for a, b in PHW], 1)
    return out

## figs10d-slope-no-difference

> Tracked OB-p and mPFC-p neurons do not differ in the average slope of their session-averaged
> normalized fluorescence across the three Go/NoGo days.

Per tracked neuron and combination of valence (CS+, CS-) and phase, the mean normalized
fluorescence on each of days 1 to 3, and its least-squares slope over days; a mixed ANOVA with the
combination (8 levels) within neurons and projection between.

In [ ]:
rows = []
for subj in TRK.subject.unique():
    proj = TRK[TRK.subject == subj].proj.iloc[0]
    pm = {d: phase_means(TR[subj, d]["X"], TR[subj, d]["trials"], "valence") for d in [1, 2, 3]}
    n = TRK[TRK.subject == subj].shape[0]
    for v in ["reward", "none"]:
        for pi, ph in enumerate(PH):
            Y = np.stack([pm[d][v][:, pi] for d in [1, 2, 3]], 1)
            sl = np.polyfit([1, 2, 3], Y.T, 1)[0]
            for c in range(n):
                rows.append(dict(neuron=f"{subj}-{c}", subject=subj, proj=proj, combo=f"{v}|{ph}", slope=sl[c]))
SL = pd.DataFrame(rows)
aov = pg.mixed_anova(SL, dv="slope", within="combo", between="proj", subject="neuron")
print(aov[["Source", "DF1", "DF2", "F", "p_unc"]].to_string(index=False))
print("mean slope per projection:", SL.groupby("proj").slope.mean().round(4).to_dict(), "| neurons:", SL.groupby("proj").neuron.nunique().to_dict())
print("per mouse mean slope:", SL.groupby(["proj", "subject"]).slope.mean().round(4).to_dict())
print("paper: projection p = 0.36, combination p = 0.11, interaction p = 0.0031")

**Result: reproduced.** The mixed ANOVA on slopes gives projection F = 0.88, p = 0.35 (paper 0.36), combination p = 0.043 (paper 0.11) and interaction p = 0.0035 (paper 0.0031). The mean slopes are -0.006 (OB-p) and 0.008 (mPFC-p) per day.

## figs10e-sparseness-slope-no-difference

> Changes in lifetime sparseness across sessions show no clear effect of projection target.

Per tracked neuron, phase and day, lifetime sparseness over the 4 CS of binarized activity in
that phase (the odor phase with the backward exclusion, the others without, as in the Methods;
both window readings); slope over days 1 to 3; mixed ANOVA with phase within and projection
between. A neuron never active in a phase on some day has no sparseness there and is left out
(count printed).

In [ ]:
def sparseness_slopes(reading, seed, fill=None):
    """Per tracked neuron and phase, the slope over days 1 to 3 of lifetime sparseness over the 4 CS."""
    rng_b = np.random.default_rng(seed)
    rows = []
    for subj in TRK.subject.unique():
        proj = TRK[TRK.subject == subj].proj.iloc[0]
        sess = {s["day"]: s for s in SES if s["subject"] == subj}
        Bs = {d: [pcx.binarize(TR[subj, d]["X_all"], t_g, TR[subj, d]["fs"], (a, b), a == 0, rng_b, reading=reading,
                               trial_mask=TR[subj, d]["good"])[:, sess[d]["keep"]] for a, b in PHW] for d in [1, 2, 3]}
        for pi, ph in enumerate(PH):
            vals = []
            for d in [1, 2, 3]:
                ann = TR[subj, d]["trials"].stim_annotation.to_numpy()
                r = np.stack([Bs[d][pi][:, ann == a].mean(1) for a in ["go1", "go2", "nogo1", "nogo2"]], 1)
                v = pcx.vinje_gallant(r, 1)
                vals.append(v if fill is None else np.nan_to_num(v, nan=fill))
            V = np.stack(vals, 1)
            for c in range(V.shape[0]):
                rows.append(dict(neuron=f"{subj}-{c}", subject=subj, proj=proj, phase=ph,
                                 slope=np.polyfit([1, 2, 3], V[c], 1)[0] if np.isfinite(V[c]).all() else np.nan))
    return pd.DataFrame(rows)

def proj_bound(L):
    """OB-p minus mPFC-p difference of per-neuron mean slope (over phases), with a Welch 95% interval."""
    m = L.dropna(subset=["slope"]).groupby(["neuron", "proj"]).slope.mean().reset_index()
    a, b = m[m.proj == "OB"].slope.to_numpy(), m[m.proj == "mPFC"].slope.to_numpy()
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    dfw = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    h = stats.t.ppf(0.975, dfw) * np.sqrt(va + vb)
    return f"difference {a.mean() - b.mean():+.4f} per day, 95% interval {a.mean() - b.mean() - h:+.4f} to {a.mean() - b.mean() + h:+.4f} " \
           f"(SD of neuron slopes {np.concatenate([a, b]).std(ddof=1):.4f}; n = {len(a)} and {len(b)})"

for reading in ["start", "centre"]:
    LSS = sparseness_slopes(reading, 4700)
    bad = LSS[LSS.slope.isna()].neuron.unique()
    LSc = LSS[~LSS.neuron.isin(bad)]
    print(f"reading {reading}: neurons with an undefined sparseness on some phase and day: {len(bad)} of {LSS.neuron.nunique()}; "
          f"kept {LSc.groupby('proj').neuron.nunique().to_dict()}")
    aov_e = pg.mixed_anova(LSc, dv="slope", within="phase", between="proj", subject="neuron")
    print(aov_e[["Source", "DF1", "DF2", "F", "p_unc"]].to_string(index=False))
    print("   mean slope per projection:", LSc.groupby("proj").slope.mean().round(4).to_dict())
    print("   bound:", proj_bound(LSc))
    for fill in [0.0, 1.0]:
        LSf = sparseness_slopes(reading, 4700, fill)
        a_f = pg.mixed_anova(LSf, dv="slope", within="phase", between="proj", subject="neuron").set_index("Source")
        print(f"   variant, undefined sparseness set to {fill:.0f} (all 87 neurons): projection p = {a_f.loc['proj', 'p_unc']:.4f}, "
              f"phase p = {a_f.loc['phase', 'p_unc']:.4f}, interaction p = {a_f.loc['Interaction', 'p_unc']:.4f}")
        print("      bound:", proj_bound(LSf))
print("paper: projection p = 0.364, phase p = 0.0491, interaction p = 0.765")


**Result: reproduced, resting on an assumption.** Most tracked neurons are never active in some phase on some day, so their sparseness is undefined. Leaving those neurons out gives projection p = 0.29 (start reading, 16 neurons kept) and p = 0.48 (centre reading, 8 kept). With all 87 neurons and an undefined sparseness set to 1, p = 0.38 and 0.45; set to 0, p = 0.082 and 0.048. Only that last variant finds a projection effect. The test has little power: the OB-p minus mPFC-p difference of mean slope is +0.032 per day with a 95% interval of -0.022 to +0.087 (start, 16 neurons; neuron slopes have SD 0.058), and +0.014 (-0.018 to +0.046) with all 87 neurons and undefined values set to 1. So the data do not exclude a difference of several hundredths per day; the absence of an effect is only shown in the sense of the criterion.

## fig5c-cv-higher-obp

> Tracked OB-p neurons show a higher coefficient of variation of session-averaged fluorescence
> across days than mPFC-p neurons.

Per tracked neuron and combination, the CV over days 1 to 3 of the session-averaged normalized
fluorescence (SD with ddof 0, a standard choice). The combination is odor (CS) by phase, 16
levels, as the amended criterion has it (questions/000785.yaml: the Results name "combinations
of odors and task phases", and Figure S10F compares CV "across different odors and phases");
valence by phase, 8 levels, is shown as a variant only. Readings of the CV: (c) SD over the
absolute mean or (d) SD over the mean. Mixed ANOVA, combination within, projection between.
Per-condition two-sided Mann-Whitney U with Benjamini-Hochberg correction (Figure S10F) reported.

In [ ]:
CVR = []
# The criterion uses odor (CS) by phase, 16 levels; valence by phase, 8 levels, is a variant only.
for comb, by in [("main: CS x phase", "stim_annotation"), ("variant: valence x phase", "valence")]:
    rows = []
    for subj in TRK.subject.unique():
        proj = TRK[TRK.subject == subj].proj.iloc[0]
        pm = {d: phase_means(TR[subj, d]["X"], TR[subj, d]["trials"], by) for d in [1, 2, 3]}
        for lev in pm[1]:
            for pi, ph in enumerate(PH):
                Y = np.stack([pm[d][lev][:, pi] for d in [1, 2, 3]], 1)
                for c in range(Y.shape[0]):
                    rows.append(dict(neuron=f"{subj}-{c}", subject=subj, proj=proj, combo=f"{lev}|{ph}",
                                     sd=Y[c].std(), mean=Y[c].mean()))
    D = pd.DataFrame(rows)
    print(f"{comb}: {D.neuron.nunique()} neurons, {D.combo.nunique()} combinations, "
          f"{(D['mean'] < 0).sum()} of {len(D)} means negative, {(D['mean'].abs() < 0.01).sum()} with |mean| < 0.01, "
          f"mean range {D['mean'].min():.3f} to {D['mean'].max():.3f}")
    for cvn, f in [("c: SD/|mean|", lambda d: d.sd / d["mean"].abs()), ("d: SD/mean", lambda d: d.sd / d["mean"])]:
        D["cv"] = f(D)
        aov_c = pg.mixed_anova(D, dv="cv", within="combo", between="proj", subject="neuron")
        a = aov_c.set_index("Source")
        med = D.groupby("proj").cv.median()
        pcs = [pcx.mwu(D[(D.combo == cb) & (D.proj == "OB")].cv, D[(D.combo == cb) & (D.proj == "mPFC")].cv) for cb in D.combo.unique()]
        rej, qv = pg.multicomp(np.array(pcs), method="fdr_bh")
        CVR.append(dict(reading=comb + ", " + cvn, p_projection=a.loc["proj", "p_unc"], p_combo=a.loc["combo", "p_unc"],
                        p_interaction=a.loc["Interaction", "p_unc"], median_OB=med["OB"], median_mPFC=med["mPFC"],
                        mean_OB=D[D.proj == "OB"].cv.mean(), mean_mPFC=D[D.proj == "mPFC"].cv.mean(),
                        n_conditions_BH=int(rej.sum()), n_conditions=len(pcs)))
        if cvn.startswith("c"):
            pm_ = {k: round(float(v), 3) for k, v in D.groupby(["proj", "subject"]).cv.median().items()}
            print("  median CV per mouse (" + cvn + "):", pm_)
            print("  CV range", round(float(D.cv.min()), 3), "to", round(float(D.cv.max()), 3))
CVR = pd.DataFrame(CVR)
print(CVR.round(4).to_string(index=False))
ok = (CVR.p_projection < 0.05) & (CVR.median_OB > CVR.median_mPFC)
print("projection p < 0.05 with OB median higher, per reading:", dict(zip(CVR.reading, ok)))
print("criterion (CS x phase under both CV readings):", bool(ok[CVR.reading.str.startswith("main")].all()))
print("paper: projection p = 0.0127, combination p = 0.0372, interaction p = 0.43")

**Result: reproduced.** Under the criterion's combination, odor (CS) by phase with 16 levels, the projection effect has p = 0.0115 (paper 0.0127) with the OB-p median CV higher (0.230 against 0.193); the combination effect has p = 0.012 (paper 0.037) and the interaction p = 0.46 (paper 0.43). Every session-averaged mean is positive (0.198 to 1.677), so the two CV readings give identical values. Each of the 3 OB-p mice has a higher median CV (0.223 to 0.265) than each of the 3 mPFC-p mice (0.187 to 0.212). The valence-by-phase variant, no longer part of the criterion, gives projection p = 0.067 with the OB-p median still higher (0.162 against 0.145). No per-condition test survives Benjamini-Hochberg correction (0 of 16), as in Figure S10F.

## fig5e-transferability

> Identity decoding transfers across sessions comparably in tracked OB-p and mPFC-p ensembles,
> whereas generalized valence decoding transfers better in mPFC-p ensembles, especially across
> distal sessions.

Per mouse, a linear SVM on its tracked projection neurons is trained on one (day, phase) and
tested on another, days 1 to 3 and the 4 phases (`pcx.transfer_matrix`: within a day, 5-fold
stratified cross-validation; across days, all trials). Only same-phase and adjacent-phase pairs
are grouped: within session, proximal (consecutive days) and distal (day 1 and day 3). Samples
of the two-sided Mann-Whitney U test (a) every qualifying entry of every mouse, (b) per mouse
means; Bonferroni over the 3 groups; aggregates mean and maximum. Matrices for valence and
correct-trial valence are computed too, for fig5f-stability.

In [ ]:
rng_t = np.random.default_rng(4800)
TM = {}
for subj in TRK.subject.unique():
    for agg in ["mean", "max"]:
        for scheme, correct in [("identity", False), ("generalized", False), ("valence", False), ("valence", True)]:
            resp, anns = [], []
            for d in [1, 2, 3]:
                v = TR[subj, d]
                m = np.ones(len(v["trials"]), bool)
                if correct:
                    m = v["trials"].outcome_relabel.isin(["hit", "correct_rejection"]).to_numpy()
                resp.append(pcx.phase_features(v["X"][:, m], t_g, PHW, agg))
                anns.append(v["trials"].stim_annotation.to_numpy()[m])
            name = scheme + (" correct" if correct else "")
            TM[subj, agg, name] = pcx.transfer_matrix(resp, anns, scheme if scheme != "valence" else "valence", rng_t)
print("matrices:", len(TM), "| shape", next(iter(TM.values())).shape,
      "| entries not fitted (a class missing from training):", {k: int(np.isnan(v).sum()) for k, v in TM.items() if np.isnan(v).any()})
PROJ = TRK.drop_duplicates("subject").set_index("subject").proj

def qualifying(M):
    out = {"within": [], "proximal": [], "distal": []}
    for i in range(3):
        for j in range(3):
            for p in range(4):
                for q in range(4):
                    if abs(p - q) > 1:
                        continue
                    k = {0: "within", 1: "proximal", 2: "distal"}[abs(i - j)]
                    out[k].append(M[i * 4 + p, j * 4 + q])
    return out

In [ ]:
res5e = []
for name in ["identity", "generalized"]:
    for agg in ["mean", "max"]:
        Q = {subj: qualifying(TM[subj, agg, name]) for subj in PROJ.index}
        for grp in ["within", "proximal", "distal"]:
            ob_a = np.concatenate([Q[s][grp] for s in PROJ.index if PROJ[s] == "OB"])
            mp_a = np.concatenate([Q[s][grp] for s in PROJ.index if PROJ[s] == "mPFC"])
            ob_b = [np.mean(Q[s][grp]) for s in PROJ.index if PROJ[s] == "OB"]
            mp_b = [np.mean(Q[s][grp]) for s in PROJ.index if PROJ[s] == "mPFC"]
            res5e.append(dict(scheme=name, agg=agg, group=grp, OB_mean=100 * ob_a.mean(), mPFC_mean=100 * mp_a.mean(),
                              n_OB=len(ob_a), n_mPFC=len(mp_a), p_entries_bonf=min(1, 3 * pcx.mwu(ob_a, mp_a)),
                              p_mice_bonf=min(1, 3 * pcx.mwu(ob_b, mp_b)),
                              per_mouse=dict((s, round(100 * np.mean(Q[s][grp]), 1)) for s in PROJ.index)))
R5E = pd.DataFrame(res5e)
print(R5E.drop(columns="per_mouse").round(4).to_string(index=False))
print(R5E[["scheme", "agg", "group", "per_mouse"]].to_string(index=False))
for agg in ["mean", "max"]:
    for samp in ["p_entries_bonf", "p_mice_bonf"]:
        idn = R5E[(R5E.scheme == "identity") & (R5E["agg"] == agg)].set_index("group")
        gv = R5E[(R5E.scheme == "generalized") & (R5E["agg"] == agg)].set_index("group")
        ok = (idn.loc["proximal", samp] >= 0.05) and (idn.loc["distal", samp] >= 0.05) and \
             (gv.loc["distal", "mPFC_mean"] > gv.loc["distal", "OB_mean"]) and (gv.loc["distal", samp] < 0.05)
        print(f"{agg}, {samp}: identity proximal and distal not different, generalized distal mPFC higher with p < 0.05: {ok}")
print("\nreading (a) leaving out one mouse at a time (corrected p of OB-p against mPFC-p; mPFC-p minus OB-p mean, points):")
rows = []
for out in PROJ.index:
    keep_m = [s for s in PROJ.index if s != out]
    r = dict(left_out=f"sub-{out} ({PROJ[out]})")
    for name, grp in [("identity", "proximal"), ("identity", "distal"), ("generalized", "distal")]:
        for agg in ["mean", "max"]:
            Q = {s: qualifying(TM[s, agg, name]) for s in keep_m}
            ob = np.concatenate([Q[s][grp] for s in keep_m if PROJ[s] == "OB"])
            mp = np.concatenate([Q[s][grp] for s in keep_m if PROJ[s] == "mPFC"])
            r[f"{name[:3]} {grp} {agg}"] = f"{min(1, 3 * pcx.mwu(ob, mp)):.2g} ({100 * (mp.mean() - ob.mean()):+.1f})"
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With every qualifying entry as a sample (reading a), identity transfer does not differ between OB-p and mPFC-p for proximal (30.0 against 30.2%, corrected p = 1.0) or distal sessions (31.1 against 30.0%, p = 1.0). Distal generalized valence is higher for mPFC-p (55.9 against 48.3%, corrected p < 0.0001). The maximum aggregate gives the same. With per-mouse means (3 against 3 mice) no comparison can reach significance (smallest corrected p = 0.6). Reading (a) treats correlated entries of 3 mice per group as independent. Leaving out one mouse at a time, distal generalized valence stays higher for mPFC-p (corrected p <= 0.0019 under both aggregates), but without sub-2400 OB-p identity is higher for distal sessions (corrected p = 0.0062, mean aggregate), so the identity part holds in 5 of the 6 leave-one-out sets. Within session, OB-p identity is higher under the maximum (37.1 against 32.7%, p = 0.0005), in the direction of the paper's within-session result.

## fig5f-stability

> The temporal structure of valence coding across sessions is more stable in tracked mPFC-p than
> OB-p ensembles, while odor identity coding is not.

From the per-mouse transfer matrices: each 4 by 4 block (train day, test day) flattened, Pearson
correlations between pairs of blocks. Readings: pairs (a) all pairs of the 9 blocks or (b) pairs
sharing the training day; samples (c) every correlation of every mouse or (d) per mouse means.
Two-sided Mann-Whitney U of OB-p against mPFC-p, Bonferroni over the 4 schemes.

In [ ]:
def block_corrs(M, pairing):
    blocks = {(i, j): M[i * 4:(i + 1) * 4, j * 4:(j + 1) * 4].ravel() for i in range(3) for j in range(3)}
    keys = list(blocks)
    out = []
    for a in range(len(keys)):
        for b in range(a + 1, len(keys)):
            if pairing == "b" and keys[a][0] != keys[b][0]:
                continue
            x, y = blocks[keys[a]], blocks[keys[b]]
            ok = np.isfinite(x) & np.isfinite(y)          # entries that could not be fitted are left out
            x, y = x[ok], y[ok]
            out.append(np.corrcoef(x, y)[0, 1] if ok.sum() > 2 and x.std() > 0 and y.std() > 0 else np.nan)
    return np.array(out)

res5f = []
for agg in ["mean", "max"]:
    for pairing in ["a", "b"]:
        for samp in ["c", "d"]:
            for name in ["identity", "valence", "generalized", "valence correct"]:
                C = {s: block_corrs(TM[s, agg, name], pairing) for s in PROJ.index}
                if samp == "c":
                    ob = np.concatenate([C[s] for s in PROJ.index if PROJ[s] == "OB"])
                    mp = np.concatenate([C[s] for s in PROJ.index if PROJ[s] == "mPFC"])
                else:
                    ob = np.array([np.nanmean(C[s]) for s in PROJ.index if PROJ[s] == "OB"])
                    mp = np.array([np.nanmean(C[s]) for s in PROJ.index if PROJ[s] == "mPFC"])
                res5f.append(dict(agg=agg, reading=pairing + samp, scheme=name, OB_mean_r=np.nanmean(ob), mPFC_mean_r=np.nanmean(mp),
                                  n_OB=int(np.isfinite(ob).sum()), n_mPFC=int(np.isfinite(mp).sum()), p_bonf=min(1, 4 * pcx.mwu(ob, mp))))
R5F = pd.DataFrame(res5f)
print(R5F.round(4).to_string(index=False))
for (agg, rd), d in R5F.groupby(["agg", "reading"]):
    d = d.set_index("scheme")
    ok = all((d.loc[n, "mPFC_mean_r"] > d.loc[n, "OB_mean_r"]) and (d.loc[n, "p_bonf"] < 0.05) for n in ["valence", "generalized", "valence correct"]) \
        and d.loc["identity", "p_bonf"] >= 0.05
    print(f"{agg}, reading {rd}: valence schemes mPFC higher with p < 0.05 and identity not different: {ok}")

**Result: not reproduced.** Under no combination of readings do all three valence schemes differ, which the criterion requires. With all block pairs, every correlation as a sample and the maximum aggregate, mPFC-p is more stable for generalized valence (r 0.31 against 0.15, corrected p = 0.0015) and correct-trial valence (0.30 against 0.12, p = 0.0014), but not for valence (0.35 against 0.29, p = 0.13), and identity does not differ (p = 1.0). mPFC-p has the higher mean correlation in 22 of the 24 valence comparisons. With per-mouse means no test is significant.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4c-behavior-learning | not reproduced (day effect not significant, F = 1.57) |
| fig4c-anticipatory-licking | reproduced |
| fig4def-identity-decoding-gng | reproduced |
| fig4g-valence-any | reproduced |
| fig4h-valence-timing | not reproduced (OB-p above chance in act-late) |
| fig4ij-valence-correct-trials | reproduced, resting on an assumption (correct trials balanced over the 4 CS, 12 trials) |
| figs8-generalized-valence-correct | not reproduced |
| discussion-differences-grow | reproduced, resting on an assumption (sub-668 left out) |
| tracked-neuron-counts | reproduced |
| figs10d-slope-no-difference | reproduced |
| figs10e-sparseness-slope-no-difference | reproduced, resting on an assumption |
| fig5c-cv-higher-obp | reproduced (odor by phase, as the amended criterion has it) |
| fig5e-transferability | reproduced |
| fig5f-stability | not reproduced (unconstrained valence not more stable) |

How the analysis differs from the paper's. The main analyses leave out sub-668, the fourth
mPFC mouse, which the paper's 7 mice do not seem to include (an assumption). Pseudo-population
trials are matched by CS and order and truncated to the smallest count, which leaves 9 trials
for one CS on day 3 (shuffled-label identity accuracy 26 to 29% against the paper's 25%). The
correct-trial table matched that way is 77% CS+, so its valence chance is about 66%, not the
paper's 50%; the correct-trial claims are judged on a table balanced over the 4 CS (3 trials
each, chance about 50%), an assumption. Within-session transfer uses 5-fold
stratified cross-validation, which the paper does not state. Neurons without a defined
lifetime sparseness are left out of the sparseness slopes (variants shown).

What was found about the dataset. The Go/NoGo experiment has 8 mice, one more mPFC-tagged
mouse (sub-668, without tracking identifiers and with 24 to 58 cells per session) than the
paper's 7. Its day 3 performance (90%) passes the paper's exclusion threshold, so the dandiset
does not say why it is not in the paper. The stated imaging rate of the GNG_2023b files is 50 or
100 Hz where the timestamps give 4.51 to 4.52 Hz. The tracking identifiers reproduce the paper's
46 and 41 tracked projection neurons exactly. Two identity sessions (sub-164 and sub-8, both
2020-06-24) each hold a plane whose mean image nearly repeats another plane of the same file
(r = 0.994 and 0.996); these are planes without red cells, which the paper's code skips, so the
duplication is consistent with the code's rule but is not marked in the files. No recording is
deposited twice across files.